In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error


N = 20000
np.random.seed(42)

order_id = np.arange(1, N+1)
time_of_day = np.random.randint(0, 24, size=N)
order_size = np.random.randint(1, 6, size=N)
cook_complexity = np.random.choice([1, 2, 3], size=N, p=[0.5, 0.35, 0.15])


rush_raw = np.random.gamma(shape=2, scale=0.8, size=N)
rush_index = np.clip(rush_raw, 0, 3.5)

KPT_true = (
    10
    + 2 * (cook_complexity - 1)
    + 0.8 * order_size
    + 3 * np.sin(time_of_day * np.pi / 24)
    + 4 * rush_index
    + np.random.normal(0, 2, size=N)
)

bias = np.where(
    rush_index < 1.5,
    np.random.uniform(3, 7, size=N),
    np.random.uniform(8, 15, size=N)
)
FOR_merchant = KPT_true - bias

FOR_cubby = KPT_true + np.random.normal(0, 0.5, size=N)

X_base = pd.DataFrame({
    "order_size": order_size,
    "cook_complexity": cook_complexity,
    "time_of_day": time_of_day
})
y_base_target = FOR_merchant

lr_base = LinearRegression()
lr_base.fit(X_base, y_base_target)

pred_base = lr_base.predict(X_base)
rmse_base = np.sqrt(mean_squared_error(KPT_true, pred_base))
p90_base = np.percentile(np.abs(KPT_true - pred_base), 90)

X_hybrid = pd.DataFrame({
    "order_size": order_size,
    "cook_complexity": cook_complexity,
    "time_of_day": time_of_day,
    "rush_index": rush_index,
    "rush_index_sq": rush_index**2
})
y_hybrid_target = FOR_cubby

lr_hybrid = LinearRegression()
lr_hybrid.fit(X_hybrid, y_hybrid_target)

pred_hybrid = lr_hybrid.predict(X_hybrid)
rmse_hybrid = np.sqrt(mean_squared_error(KPT_true, pred_hybrid))
p90_hybrid = np.percentile(np.abs(KPT_true - pred_hybrid), 90)

print(f"--- ERROR METRICS ---")
print(f"Baseline RMSE: {rmse_base:.2f} mins | Baseline P90 Error: {p90_base:.2f} mins")
print(f"Hybrid RMSE:   {rmse_hybrid:.2f} mins | Hybrid P90 Error:   {p90_hybrid:.2f} mins")


plt.style.use('ggplot')

plt.figure(figsize=(6,4))
plt.bar(["Baseline (Current)", "Hybrid (IoT + Rush Index)"], [rmse_base, rmse_hybrid], color=['#e74c3c', '#2ecc71'])
plt.title("Average Prediction Error (RMSE)")
plt.ylabel("Minutes")
plt.savefig("rmse_comparison.png", bbox_inches='tight')
plt.close()

# Plot B: P90 Error Comparison (The Money Metric for Rider Wait Time)
plt.figure(figsize=(6,4))
plt.bar(["Baseline (Current)", "Hybrid (IoT + Rush Index)"], [p90_base, p90_hybrid], color=['#c0392b', '#27ae60'])
plt.title("Worst-Case Scenario Errors (90th Percentile)")
plt.ylabel("Minutes of Error")
plt.savefig("p90_comparison.png", bbox_inches='tight')
plt.close()

time_steps = np.arange(50)
sim_rush = np.exp(-0.5 * ((time_steps - 25) / 5)**2) * 3.2 + np.random.uniform(0, 0.3, 50) # Bell curve spike

sim_X_base = pd.DataFrame({
    "order_size": np.full(50, 2),
    "cook_complexity": np.full(50, 2),
    "time_of_day": np.full(50, 20)
})

sim_X_hybrid = sim_X_base.copy()
sim_X_hybrid["rush_index"] = sim_rush
sim_X_hybrid["rush_index_sq"] = sim_rush**2

sim_true_kpt = 10 + 2*(2-1) + 0.8*2 + 3*np.sin(20*np.pi/24) + 4*sim_rush + np.random.normal(0, 1, 50)

sim_pred_base = lr_base.predict(sim_X_base)
sim_pred_hybrid = lr_hybrid.predict(sim_X_hybrid)

plt.figure(figsize=(10,5))
plt.plot(time_steps, sim_true_kpt, label="True KPT (Actual Wait Time)", color='black', linewidth=3)
plt.plot(time_steps, sim_pred_base, label="Baseline Prediction (Blind to Rush)", color='#e74c3c', linestyle='dashed', linewidth=2)
plt.plot(time_steps, sim_pred_hybrid, label="Hybrid Prediction (IoT Enabled)", color='#2ecc71', linestyle='dashed', linewidth=2)
plt.fill_between(time_steps, 0, sim_rush * 10, color='gray', alpha=0.15, label="Ambient Rush Level (Background)")

plt.title("Simulated Friday Night Peak: KPT Tracking")
plt.xlabel("Order Sequence (Time)")
plt.ylabel("Preparation Time (Minutes)")
plt.legend(loc="upper left")
plt.savefig("friday_night_spike.png", bbox_inches='tight')
plt.close()

print("All presentation plots saved successfully.")

--- ERROR METRICS ---
Baseline RMSE: 9.06 mins | Baseline P90 Error: 14.25 mins
Hybrid RMSE:   2.21 mins | Hybrid P90 Error:   3.63 mins
All presentation plots saved successfully.
